# Chapter 13 — Exercise Solutions## How LLMs Are Trained[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)

### Solution 13.1: Token-Level Log-Probability Computation

In [ ]:
import torch
import numpy as np

def compute_sequence_logprob(model, tokenizer, prompt: str, response: str) -> dict:
    """
    Compute per-token and total log-probabilities for a (prompt, response) pair.
    This is the core computation used in DPO and RLHF reward computation.
    """
    full_text = prompt + response
    prompt_ids = tokenizer(prompt, return_tensors='pt')['input_ids']
    full_ids   = tokenizer(full_text, return_tensors='pt')['input_ids']

    model.eval()
    with torch.no_grad():
        logits = model(full_ids).logits  # [1, seq_len, vocab_size]

    log_probs = torch.log_softmax(logits, dim=-1)  # [1, seq_len, vocab_size]
    n_prompt = prompt_ids.shape[1]

    # Extract log-probs for each response token
    response_token_ids = full_ids[0, n_prompt:]
    per_token_logprobs = []
    for i, token_id in enumerate(response_token_ids):
        pos = n_prompt + i - 1  # logit at pos t predicts token at pos t+1
        if pos >= 0:
            lp = log_probs[0, pos, token_id].item()
            per_token_logprobs.append({
                'token': tokenizer.decode([token_id]),
                'token_id': token_id.item(),
                'log_prob': lp,
                'prob': np.exp(lp)
            })

    total_logprob = sum(t['log_prob'] for t in per_token_logprobs)
    return {
        'per_token': per_token_logprobs,
        'total_log_prob': total_logprob,
        'mean_log_prob': total_logprob / max(len(per_token_logprobs), 1),
        'perplexity': np.exp(-total_logprob / max(len(per_token_logprobs), 1))
    }

# Demonstration without real model (show the structure)
print("Function signature and expected output:")
print("""
compute_sequence_logprob(model, tokenizer, 
    prompt='What is 2+2?',
    response=' The answer is 4.'
)
→ {
    'per_token': [
        {'token': ' The',    'log_prob': -2.3, 'prob': 0.10},
        {'token': ' answer', 'log_prob': -1.8, 'prob': 0.17},
        {'token': ' is',     'log_prob': -0.5, 'prob': 0.61},
        {'token': ' 4',      'log_prob': -0.3, 'prob': 0.74},
        {'token': '.',       'log_prob': -0.8, 'prob': 0.45},
    ],
    'total_log_prob': -5.7,
    'mean_log_prob':  -1.14,
    'perplexity':      3.13
}
""")
print("In DPO: loss uses (total_log_prob_policy - total_log_prob_reference)")
print("  → chosen_logps - chosen_ref_logps > rejected_logps - rejected_ref_logps")
print("  → policy assigns HIGHER relative probability to chosen than reference does")

# YOUR TURN: Using distilgpt2 or gpt2:
# 1. Compute log_prob for a correct and incorrect answer to a simple question
# 2. Verify: P(correct) > P(incorrect) for a well-calibrated model
# 3. Compute the DPO log-ratio: log_p_policy - log_p_reference


### Solution 13.2: SFT Token Masking

In [ ]:
import torch

def create_sft_labels(input_ids: torch.Tensor, prompt_length: int,
                      ignore_index: int = -100) -> torch.Tensor:
    """
    Create labels for SFT training: mask prompt tokens with ignore_index.
    Loss is computed only on response tokens.
    """
    labels = input_ids.clone()
    labels[:, :prompt_length] = ignore_index  # mask prompt
    return labels

def demonstrate_masking():
    # Simulate tokenized (prompt + response)
    prompt_tokens   = [1, 234, 567, 89, 12]   # 5 prompt tokens
    response_tokens = [43, 21, 8, 99, 2]       # 5 response tokens
    eos_token = 2

    input_ids = torch.tensor([prompt_tokens + response_tokens])  # shape [1, 10]
    labels    = create_sft_labels(input_ids, prompt_length=len(prompt_tokens))

    print("Input IDs:", input_ids[0].tolist())
    print("Labels:   ", labels[0].tolist())
    print()
    print("Positions -100 (masked/ignored):", (labels[0] == -100).sum().item(), "tokens")
    print("Positions with real labels:      ", (labels[0] != -100).sum().item(), "tokens")
    print()
    print("During training:")
    print("  loss = cross_entropy(logits, labels, ignore_index=-100)")
    print("  Only response positions contribute gradients")
    print("  Prompt positions are FREE — the model sees them but doesn't learn to predict them")
    print()
    print("Why this matters:")
    print("  If we computed loss on prompt too, model would learn to predict the question")
    print("  That wastes capacity and can hurt instruction-following quality")

demonstrate_masking()

# Verify with actual HuggingFace behaviour
print("\n--- HuggingFace SFT Implementation ---")
print("In TRL's SFTTrainer:")
print("  dataset_text_field='text'  → auto-masks with DataCollatorForSeq2Seq")
print("  OR: manually set labels = [-100]*prompt_len + response_ids")
print()
print("In practice (from TRL source code):")
print("  for i, token_id in enumerate(input_ids):")
print("      labels[i] = token_id if i >= prompt_length else -100")

# YOUR TURN: Compute the perplexity difference between:
# a) Model trained with prompt masking (SFT)
# b) Model trained without prompt masking (full sequence loss)
# Which should have lower perplexity on responses? Why?
